In [1]:

import torch

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA is available")
else:
    print("No GPU detected — CPU mode will be slower.")
    print("To check: Runtime > Change runtime type > T4 GPU, if available.")


GPU: Tesla T4
CUDA is available


In [4]:

%pip install -q -U transformers accelerate gradio


In [5]:

import torch
from transformers import pipeline

MODEL_ID = "mohammedaly22/whisper-large-v3-turbo-egyptian-code-switching"

device = 0 if torch.cuda.is_available() else -1
dtype = torch.float16 if torch.cuda.is_available() else torch.float32

asr = pipeline(
    "automatic-speech-recognition",
    model=MODEL_ID,
    device=device,
    torch_dtype=dtype,
    chunk_length_s=30,
)

print("Model loaded:", MODEL_ID)
print("Device:", torch.cuda.get_device_name(0)
      if torch.cuda.is_available() else "CPU")


config.json:   0%|          | 0.00/1.23k [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.62GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.93M [00:00<?, ?B/s]

processor_config.json:   0%|          | 0.00/410 [00:00<?, ?B/s]

[transformers] Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on its own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


Model loaded: mohammedaly22/whisper-large-v3-turbo-egyptian-code-switching
Device: Tesla T4


In [7]:

import gradio as gr

def transcribe(audio_path, language):
    if audio_path is None:
        return "Please record something first."

    options = {"task": "transcribe"}

    # Arabic is appropriate for Egyptian Arabic and MSA.
    # The fine-tuned model is intended to preserve English
    # words when speakers switch between Arabic and English.
    if language == "Arabic / Egyptian Arabic":
        options["language"] = "arabic"
    elif language == "English":
        options["language"] = "english"

    try:
        result = asr(
            audio_path,
            generate_kwargs=options
        )
        return result["text"].strip()
    except Exception as error:
        return f"Error: {type(error).__name__}: {error}"


app = gr.Interface(
    fn=transcribe,
    inputs=[
        gr.Audio(
            sources=["microphone"],
            type="filepath",
            label="🎙️ Speak here"
        ),
        gr.Dropdown(
            choices=[
                "Auto-detect",
                "Arabic / Egyptian Arabic",
                "English"
            ],
            value="Arabic / Egyptian Arabic",
            label="Speech language"
        )
    ],
    outputs=gr.Textbox(
        label="Transcription",
        lines=8
    ),
    title="Arabic + English Speech Recognition",
    description=(
        "Record Egyptian Arabic, formal Arabic, English, "
        "or a mix of Arabic and English. Stop recording "
        "to receive the transcription."
    )
)

app.launch(share=True, debug=True)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://a39829f27c403ee400.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


[transformers] Passing `generation_config` together with generation-related arguments=({'begin_suppress_tokens', 'suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://a39829f27c403ee400.gradio.live
